In [ ]:
# ── CELL 1: Dependencies ──────────────────────────────────────────────────
!apt-get install -y openssh-server > /dev/null 2>&1
!pip install -q ipywidgets

# colab-connect — SSH Tunnel Bridge via Cloudflare

Establishes a secure OpenSSH server inside Google Colab and exposes it through a Cloudflare Tunnel (`trycloudflare.com`). Enables interactive terminal SSH, remote execution, and local-to-cloud workflows with your Mac or AI agent directly on Google Colab hardware (GPU/CPU).

---

## Source
- **Site / tool:** OpenSSH Server + Cloudflare Tunnel (`cloudflared`)
- **How to find input:** Run `make setup-colab` locally or copy your public SSH key from `~/.ssh/id_rsa.pub`.

## What it does
1. Installs OpenSSH server and the `cloudflared` tunnel client.
2. Configures `authorized_keys` with your public SSH key.
3. Starts the SSH daemon securely on port 22.
4. Starts Cloudflare quick tunnel and captures the active `trycloudflare.com` URL.
5. Displays the active hostname and ready-to-run SSH connection commands.
6. Saves session metadata to Google Drive at `CollabMedia/utils/colab-connect/session.json`.
7. Enters a keepalive loop to keep the tunnel and runtime active.

## How to use
1. Run **Cell 1** (`apt-get install` / `pip install`) — installs OpenSSH and widgets.
2. Run **Cell 3** — displays the SSH Public Key input widget.
3. Paste the contents of your local `~/.ssh/id_rsa.pub` into the text box and click **Save Key**.
4. Run **Cell 4** — mounts Google Drive and prepares directories.
5. Run **Cell 5** — installs `cloudflared`, configures `authorized_keys`, and launches `sshd`.
6. Run **Cell 6** — starts the Cloudflare Tunnel, extracts the URL, and displays connection commands.
7. Run **Cell 7** — saves `session.json` to Google Drive and runs the keepalive loop.

## Known limitations
- Free Google Colab sessions disconnect if completely idle or if the browser tab is closed. Keep the browser tab open.
- Each time the Colab runtime starts or reconnects, a new ephemeral tunnel hostname is generated.

## Output
```
MyDrive/
└── CollabMedia/
    └── utils/
        └── colab-connect/
            └── session.json
```

In [ ]:
# ── CELL 3: SSH Public Key Widget ─────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

box_key = widgets.Textarea(
    value="",
    placeholder="Paste your public SSH key (~/.ssh/id_rsa.pub) here...",
    description="SSH PubKey:",
    layout=widgets.Layout(width="100%", height="140px")
)

btn_apply = widgets.Button(description="✓ Save Key", button_style="success")
out_status = widgets.Output()

def on_apply_clicked(b):
    with out_status:
        out_status.clear_output()
        val = box_key.value.strip()
        if not val:
            print("⚠ Please paste a valid SSH public key.")
        else:
            print(f"✓ Public key saved ({len(val)} chars). Ready for Cell 4 & 5.")

btn_apply.on_click(on_apply_clicked)
display(widgets.VBox([
    widgets.HTML("<b>Paste your local SSH public key below (from `make setup-colab` or `cat ~/.ssh/id_rsa.pub`):</b>"),
    box_key,
    btn_apply,
    out_status
]))

In [ ]:
# ── CELL 4: Setup & Drive Mount ───────────────────────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")

DEST_DIR = "/content/drive/MyDrive/CollabMedia/utils/colab-connect"
TEMP_DIR = "/content/utils/colab-connect"

os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)
print("Drive mounted. Directories prepared.")

In [ ]:
# ── CELL 5: Install cloudflared & Configure OpenSSH ─────────────────────────
import os, subprocess, urllib.request

# 1. Setup authorized_keys
ssh_dir = os.path.expanduser("~/.ssh")
os.makedirs(ssh_dir, mode=0o700, exist_ok=True)
auth_file = os.path.join(ssh_dir, "authorized_keys")

pubkey = box_key.value.strip()
if not pubkey:
    raise ValueError("SSH public key is empty! Paste your key in Cell 3 first.")

with open(auth_file, "w") as f:
    f.write(pubkey + "\n")
os.chmod(auth_file, 0o600)
print(f"✓ Authorized keys written to {auth_file}")

# 2. Configure and start SSHD
os.makedirs("/run/sshd", exist_ok=True)
sshd_config = """Port 22
PermitRootLogin prohibit-password
PubkeyAuthentication yes
AuthorizedKeysFile .ssh/authorized_keys
PasswordAuthentication no
ChallengeResponseAuthentication no
UsePAM yes
"""
os.makedirs("/etc/ssh/sshd_config.d", exist_ok=True)
with open("/etc/ssh/sshd_config.d/colab_connect.conf", "w") as f:
    f.write(sshd_config)

subprocess.run(["service", "ssh", "restart"], check=True)
print("✓ OpenSSH server is running on port 22.")

# 3. Download cloudflared if not present
cloudflared_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared_path):
    print("Downloading cloudflared binary...")
    cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(cf_url, cloudflared_path)
    os.chmod(cloudflared_path, 0o755)
    print("✓ cloudflared installed to /usr/local/bin/cloudflared")
else:
    print("✓ cloudflared already present.")

In [ ]:
# ── CELL 6: Start Cloudflare Tunnel ─────────────────────────────────────────
import subprocess, time, re

# Terminate any existing tunnel
subprocess.run(["pkill", "-f", "cloudflared tunnel"], stderr=subprocess.DEVNULL)
time.sleep(1)

log_path = "/tmp/cloudflared.log"
if os.path.exists(log_path):
    os.remove(log_path)

log_file = open(log_path, "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "ssh://localhost:22"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Starting Cloudflare Tunnel...")
tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            content = f.read()
            match = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", content)
            if match:
                tunnel_url = match.group(1)
                break

if not tunnel_url:
    print("Failed to capture Cloudflare Tunnel URL. Tunnel logs:")
    with open(log_path, "r") as f:
        print(f.read())
    raise RuntimeError("Could not establish Cloudflare Tunnel.")

print("\n" + "=" * 60)
print("🚀 COLAB SSH BRIDGE IS ACTIVE!")
print("=" * 60)
print(f"\nHostname: {tunnel_url}\n")
print("Connect from your local Mac using:")
print(f"  make connect-colab")
print(f"  (paste: {tunnel_url})")
print("\nOr direct terminal command:")
print(f"  ssh -o ProxyCommand=\"cloudflared access ssh --hostname %h\" root@{tunnel_url}")
print("\n" + "=" * 60)

In [ ]:
# ── CELL 7: Save Metadata to Drive & Keepalive ────────────────────────────
import time, json, datetime

# Save session metadata to Drive
session_data = {
    "hostname": tunnel_url,
    "user": "root",
    "updated_at": datetime.datetime.utcnow().isoformat() + "Z",
    "command": f'ssh -o ProxyCommand="cloudflared access ssh --hostname %h" root@{tunnel_url}'
}

session_file = os.path.join(DEST_DIR, "session.json")
with open(session_file, "w") as f:
    json.dump(session_data, f, indent=2)

print(f"✓ Session metadata saved to Drive: {session_file}")
print("\nKeeping tunnel alive. Do not stop this cell while connected.")
print("To disconnect: press Stop (■) on this cell.\n")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nBridge stopped by user.")
    cf_proc.terminate()
